In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Ensure numeric 'value' columns where applicable
for df in (metric, trace, log):
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Convert timestamp -> timezone-aware datetimes in Asia/Shanghai (UTC+8)
for df in (metric, trace, log, error_logs):
    # keep original integer timestamp too, but create dt column as tz-aware datetime
    df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Aggregate function to compute count, min_ts, max_ts, global P95
def aggregate_series(df, group_cols, value_col_name):
    agg = df.groupby(group_cols, as_index=False).agg(
        count=( 'value', 'count' ),
        min_ts=( 'dt', 'min' ),
        max_ts=( 'dt', 'max' ),
        p95=( 'value', lambda x: x.quantile(0.95) )
    )
    # sort by count desc
    agg = agg.sort_values('count', ascending=False).reset_index(drop=True)
    return agg

metric_agg = aggregate_series(metric, ['cmdb_id', 'kpi_name'], 'value')
trace_agg = aggregate_series(trace, ['cmdb_id', 'trace_name'], 'value')
log_agg = aggregate_series(log, ['cmdb_id', 'log_name'], 'value')

# Top 20 rows for each (compact)
metric_top20 = metric_agg.head(20)
trace_top20 = trace_agg.head(20)
log_top20 = log_agg.head(20)

# Unique cmdb_id values across all files
all_cmdb_ids = pd.Series(
    sorted(
        pd.unique(
            pd.concat([
                metric['cmdb_id'].dropna(),
                trace['cmdb_id'].dropna(),
                log['cmdb_id'].dropna(),
                error_logs['cmdb_id'].dropna()
            ]).astype(str)
        )
    )
).reset_index(drop=True)
all_cmdb_ids.name = 'cmdb_id'
unique_cmdb_df = all_cmdb_ids.to_frame()

# Error logs: top 20 rows (timestamp, cmdb_id, message) sorted by timestamp desc (most recent)
error_top20 = error_logs.sort_values('timestamp', ascending=False).loc[:, ['dt', 'cmdb_id', 'message']].head(20)
error_top20 = error_top20.rename(columns={'dt':'timestamp'})
error_unique_cmdb_count = error_logs['cmdb_id'].nunique(dropna=True)

# Display compact results (variables returned)
metric_top20, trace_top20, log_top20, unique_cmdb_df, error_top20, error_unique_cmdb_count

```
Out[1]:
```
```python
# Summarize results from previous aggregations (uses existing variables in the kernel)
metric_rows = metric_top20.shape[0]
trace_rows = trace_top20.shape[0]
log_rows = log_top20.shape[0]

metric_count_max = int(metric_top20['count'].max()) if metric_rows>0 else 0
trace_count_max = int(trace_top20['count'].max()) if trace_rows>0 else 0
log_count_max = int(log_top20['count'].max()) if log_rows>0 else 0

metric_time_min = metric_top20['min_ts'].min() if metric_rows>0 else None
metric_time_max = metric_top20['max_ts'].max() if metric_rows>0 else None

trace_time_min = trace_top20['min_ts'].min() if trace_rows>0 else None
trace_time_max = trace_top20['max_ts'].max() if trace_rows>0 else None

log_time_min = log_top20['min_ts'].min() if log_rows>0 else None
log_time_max = log_top20['max_ts'].max() if log_rows>0 else None

unique_cmdb_count = int(unique_cmdb_df.shape[0])
error_rows = error_top20.shape[0]
error_unique_cmdb_count = int(error_unique_cmdb_count)

summary = (
    f"Summary of aggregations (all times in UTC+8):\n"
    f"- Metric aggregates: showing up to {metric_rows} top groups (by count). "
    f"Max sample count among shown groups = {metric_count_max}. "
    f"Observed time range in shown groups ≈ {metric_time_min} to {metric_time_max}.\n"
    f"- Trace aggregates: showing up to {trace_rows} top groups. "
    f"Max sample count = {trace_count_max}. "
    f"Observed time range ≈ {trace_time_min} to {trace_time_max}.\n"
    f"- Log aggregates: showing up to {log_rows} top groups. "
    f"Max sample count = {log_count_max}. "
    f"Observed time range ≈ {log_time_min} to {log_time_max}.\n"
    f"- Unique cmdb_id values across all files: {unique_cmdb_count} (sample list available in 'unique_cmdb_df').\n"
    f"- error_logs.csv: {error_rows} rows returned (top 20). Unique cmdb_id count in error_logs = {error_unique_cmdb_count}."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count                    min_ts                    max_ts           p95
0    Redis01   redis-Redis_6379_Redis  (used_cpu_user_children)     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  0.000000e+00
1    Redis01              redis-Redis_6379_Redis  (used_memory)     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  1.764680e+07
2       IG01                            OSLinux-CPU_CPU_CPULoad     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  1.050000e+00
3       IG01                         OSLinux-CPU_CPU_CPUSysTime     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  4.033000e-01
4    Redis01  redis-Redis_6379_Redis  (total_commands_proces...     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  4.326847e+09
5    Redis01  redis-Redis_6379_Redis  (total_connections_rec...     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  2.918730e+07
6    Redis01            redis-Redis_6379_Redis  (used_cpu_user)     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  3.011181e+04
7    Mysql02                        OSLinux-CPU_CPU_CPUidleutil     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  7.579325e+01
8    Redis01         redis-Redis_6379_Redis  (used_memory_peak)     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  3.487778e+07
9    Redis01          redis-Redis_6379_Redis  (used_memory_rss)     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  1.192346e+07
10   Redis01                      redis-Redis_6379_redis server     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  1.000000e+00
11   Redis01  redis-Redis_6379_Redis  (rdb_changes_since_las...     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  9.711253e+08
12   Redis01          redis-Redis_6379_Redis  (redis_git_dirty)     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  0.000000e+00
13   Redis01     redis-Redis_6379_Redis  (rejected_connections)     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  0.000000e+00
14   Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  0.000000e+00
15   Mysql02                            OSLinux-CPU_CPU_CPULoad     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  2.825000e+00
16   Mysql02                         OSLinux-CPU_CPU_CPUSysTime     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  1.196547e+00
17   Mysql02                        OSLinux-CPU_CPU_CPUUserTime     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  1.961950e+00
18   Mysql02                             OSLinux-CPU_CPU_CPUWio     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  2.994736e+01
19  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  0.000000e+00,      cmdb_id                         trace_name  count                    min_ts                    max_ts          p95
0   dockerB2               trace.self.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00  1486.050000
1   dockerA2           trace.self.duration_mean     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     0.011401
2   dockerA2            trace.self.duration_p95     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     0.012550
3   dockerA2               trace.self.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00  1032.850000
4   dockerA1           trace.self.duration_mean     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     0.023034
5   dockerA1            trace.self.duration_p95     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     0.014550
6   dockerA1               trace.self.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00   974.750000
7       MG01           trace.self.duration_mean     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     0.133290
8       MG01            trace.self.duration_p95     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     1.046963
9       MG01      trace.from_Tomcat02.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00    90.100000
10      MG01  trace.from_Tomcat03.duration_mean     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     0.268588
11      MG01   trace.from_Tomcat03.duration_p95     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     1.253432
12      MG01      trace.from_Tomcat03.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00    86.200000
13      MG01  trace.from_Tomcat04.duration_mean     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     0.192358
14      MG01   trace.from_Tomcat04.duration_p95     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     1.192455
15      MG01      trace.from_Tomcat04.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00    97.550000
16      IG02    trace.to_Tomcat04.duration_mean     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     0.293836
17      IG02     trace.to_Tomcat04.duration_p95     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     1.276987
18      IG02        trace.to_Tomcat04.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00    91.850000
19      MG01  trace.from_Tomcat01.duration_mean     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00     0.248624,      cmdb_id          log_name  count                    min_ts                    max_ts       p95
0   apache01   log.error_count     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00    0.0000
1   apache01     log.row_count     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  337.5000
2   apache02     log.row_count     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  321.0000
3   apache02   log.error_count     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00    0.0000
4   Tomcat04     log.row_count     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00  184.0000
5   Tomcat04   log.error_count     31 2021-03-04 18:00:00+08:00 2021-03-04 18:30:00+08:00    0.0000
6   Tomcat01   log.error_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00    0.0000
7   Tomcat01     log.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00  172.6500
8   Tomcat02     log.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00  173.0000
9   Tomcat02   log.error_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00    0.0000
10  Tomcat03     log.row_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00  162.0000
11  Tomcat03   log.error_count     30 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00    0.0000
12  Tomcat03  log.gc_pause_max      9 2021-03-04 18:00:00+08:00 2021-03-04 18:29:00+08:00    0.1020
13  Tomcat01  log.gc_pause_max      8 2021-03-04 18:02:00+08:00 2021-03-04 18:26:00+08:00    0.0995
14  Tomcat02  log.gc_pause_max      8 2021-03-04 18:02:00+08:00 2021-03-04 18:26:00+08:00    0.0800
15  Tomcat04  log.gc_pause_max      8 2021-03-04 18:01:00+08:00 2021-03-04 18:26:00+08:00    0.0800,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
..       ...
24  apache02
25  dockerA1
26  dockerA2
27  dockerB1
28  dockerB2

[29 rows x 1 columns], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], 0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing 'metric' if available; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure numeric value and timezone-aware datetime in UTC+8
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) across full series (must be done BEFORE filtering)
global_p95_df = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric to incident window and restrict to specified components
components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

start = pd.Timestamp('2021-03-04 18:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-04 18:30:00').tz_localize(tz)

window_df = metric[
    (metric['dt'] >= start) & (metric['dt'] <= end) & (metric['cmdb_id'].isin(components))
].copy()

# If no rows in window, return empty DataFrame with required columns
if window_df.shape[0] == 0:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','global_p95','breach_ratio'
    ])
else:
    # Merge global P95 into window rows
    window_df = window_df.merge(global_p95_df, on=['cmdb_id','kpi_name'], how='left')

    # 3) Flag anomalies (value > global_p95) and aggregate per group
    window_df['is_anomaly'] = window_df['value'] > window_df['global_p95']

    def earliest_anom(series_dt, series_flag):
        # return smallest dt where flag True, else pd.NaT
        masked = series_dt[series_flag]
        return masked.min() if not masked.empty else pd.NaT

    agg = (
        window_df
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            anomaly_count = ('is_anomaly', 'sum'),
            earliest_anomaly_timestamp = ('dt', lambda s: earliest_anom(s, window_df.loc[s.index, 'is_anomaly'])),
            max_value_in_window = ('value', 'max'),
            global_p95 = ('global_p95', 'first')
        )
    )

    # 3b) compute breach_ratio
    def compute_breach(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp) or gp == 0:
            return None
        return (mv - gp) / gp

    agg['breach_ratio'] = agg.apply(compute_breach, axis=1)

    # 4) Sort by anomaly_count desc then breach_ratio desc, return top 20 and select columns
    result_df = (
        agg
        .sort_values(['anomaly_count','breach_ratio'], ascending=[False, False], na_position='last')
        .reset_index(drop=True)
    )

    # Keep columns order and compact timestamps
    result_df = result_df[
        ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95','breach_ratio']
    ]

# Return up to top 20 rows
result_df.head(20)

```
Out[2]:
```
# Summarize the anomaly analysis results (produced in previous step) as plain English text.
try:
    result_df
except NameError:
    summary = "No result dataframe 'result_df' found in the environment."
else:
    # compact counts and time range
    total_groups = int(result_df.shape[0])
    anomalous_groups = int((result_df['anomaly_count'] > 0).sum())
    earliest_anom = result_df['earliest_anomaly_timestamp'].min()
    latest_anom = result_df['earliest_anomaly_timestamp'].max()
    
    # prepare top lines (up to 10) showing key fields compactly
    top_display = result_df.head(10).copy()
    def fmt_num(x):
        if pd.isna(x):
            return "None"
        if isinstance(x, (int, np.integer)):
            return str(int(x))
        return f"{x:.6g}"
    lines = []
    for _, r in top_display.iterrows():
        ts = r['earliest_anomaly_timestamp']
        ts_str = ts.isoformat() if not pd.isna(ts) else "None"
        line = (
            f"{r['cmdb_id']} | {r['kpi_name']} | anomalies={int(r['anomaly_count'])} | "
            f"earliest={ts_str} | max={fmt_num(r['max_value_in_window'])} | "
            f"global_p95={fmt_num(r['global_p95'])} | breach_ratio={fmt_num(r['breach_ratio'])}"
        )
        lines.append(line)
    
    summary = (
        f"Anomaly summary for metric.csv within 2021-03-04 18:00–18:30 (UTC+8):\n"
        f"- Total (cmdb_id, kpi_name) groups examined in window: {total_groups}\n"
        f"- Groups with anomalies (value > global P95): {anomalous_groups}\n"
        f"- Earliest anomaly timestamp among reported groups: {earliest_anom}\n"
        f"- Latest earliest-anomaly timestamp among reported groups: {latest_anom}\n\n"
        f"Top groups (up to 10) sorted by anomaly_count then breach_ratio:\n"
        + "\n".join(lines)
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_p95  breach_ratio
0    Redis01                             OSLinux-CPU_CPU_CPUWio              2  2021-03-04 18:01:00+08:00         8.196000e-01  9.358600e-02      7.757720
1       IG01                             OSLinux-CPU_CPU_CPUWio              2  2021-03-04 18:00:00+08:00         2.386400e+00  2.826455e-01      7.443085
2    Redis02                        OSLinux-CPU_CPU_CPUUserTime              2  2021-03-04 18:00:00+08:00         3.138200e+00  3.925650e-01      6.994090
3    Mysql01                             OSLinux-CPU_CPU_CPUWio              2  2021-03-04 18:01:00+08:00         6.508200e+00  1.371550e+00      3.745142
4    Redis02                         OSLinux-CPU_CPU_CPUSysTime              2  2021-03-04 18:10:00+08:00         2.421300e+00  6.480100e-01      2.736516
5       IG02                             OSLinux-CPU_CPU_CPUWio              2  2021-03-04 18:01:00+08:00         3.204500e+00  9.602040e-01      2.337312
6    Mysql01                            OSLinux-CPU_CPU_CPULoad              2  2021-03-04 18:01:00+08:00         7.200000e-01  2.200000e-01      2.272727
7   apache02                             OSLinux-CPU_CPU_CPUWio              2  2021-03-04 18:00:00+08:00         1.711817e+00  6.443000e-01      1.656863
8    Redis02                             OSLinux-CPU_CPU_CPUWio              2  2021-03-04 18:10:00+08:00         3.565775e+00  1.805650e+00      0.974787
9   apache01                            OSLinux-CPU_CPU_CPULoad              2  2021-03-04 18:00:00+08:00         3.400000e-01  2.200000e-01      0.545455
10   Redis02                            OSLinux-CPU_CPU_CPULoad              2  2021-03-04 18:09:00+08:00         1.000000e-01  6.650000e-02      0.503759
11  apache01                             OSLinux-CPU_CPU_CPUWio              2  2021-03-04 18:00:00+08:00         6.313600e+00  5.086066e+00      0.241352
12   Mysql02                             OSLinux-CPU_CPU_CPUWio              2  2021-03-04 18:01:00+08:00         3.668140e+01  2.994736e+01      0.224863
13  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              2  2021-03-04 18:07:00+08:00         3.000000e+00  2.500000e+00      0.200000
14      IG01                         OSLinux-CPU_CPU_CPUSysTime              2  2021-03-04 18:05:00+08:00         4.738940e-01  4.033000e-01      0.175041
15   Redis02              redis-Redis_6379_Redis  (used_memory)              2  2021-03-04 18:07:00+08:00         1.636358e+07  1.392646e+07      0.174999
16   Mysql02                         OSLinux-CPU_CPU_CPUSysTime              2  2021-03-04 18:27:00+08:00         1.352200e+00  1.196547e+00      0.130085
17   Mysql02                        OSLinux-CPU_CPU_CPUUserTime              2  2021-03-04 18:05:00+08:00         2.184700e+00  1.961950e+00      0.113535
18   Redis01  redis-Redis_6379_Redis  (instantaneous_ops_per...              2  2021-03-04 18:10:00+08:00         1.752000e+03  1.574000e+03      0.113088
19   Redis02  redis-Redis_6379_Redis  (instantaneous_ops_per...              2  2021-03-04 18:03:00+08:00         1.284000e+03  1.154000e+03      0.112652```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse existing variables if present
tz = pytz.timezone('Asia/Shanghai')

try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Ensure numeric and timezone-aware dt column
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) across full series
global_thresh = (
    trace
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_p5=lambda x: x.quantile(0.05))
)

# 2) Filter trace to incident window and candidate cmdb_ids
candidates = ["Redis01","IG01","Redis02","Mysql01","IG02"]
start = pd.Timestamp('2021-03-04 18:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-04 18:30:00').tz_localize(tz)

window_trace = trace[
    (trace['dt'] >= start) & (trace['dt'] <= end) & (trace['cmdb_id'].isin(candidates))
].copy()

# If no data in window, return empty structured DataFrame
if window_trace.shape[0] == 0:
    trace_anom_top20 = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count_high','earliest_high_timestamp',
        'anomaly_count_low','earliest_low_timestamp','max_value_in_window','min_value_in_window',
        'global_p95','global_p5','breach_ratio_high'
    ])
else:
    # Merge global thresholds (computed before filtering) into windowed data
    window_trace = window_trace.merge(global_thresh, on=['cmdb_id','trace_name'], how='left')

    # Group and compute required aggregates
    records = []
    for (cmdb_id, trace_name), grp in window_trace.groupby(['cmdb_id','trace_name']):
        gp95 = grp['global_p95'].iloc[0]
        gp5 = grp['global_p5'].iloc[0]

        mask_high = grp['value'] > gp95
        mask_low = grp['value'] < gp5

        anomaly_count_high = int(mask_high.sum())
        anomaly_count_low = int(mask_low.sum())

        earliest_high_ts = grp.loc[mask_high, 'dt'].min() if anomaly_count_high > 0 else pd.NaT
        earliest_low_ts = grp.loc[mask_low, 'dt'].min() if anomaly_count_low > 0 else pd.NaT

        max_val = grp['value'].max()
        min_val = grp['value'].min()

        if pd.isna(gp95) or gp95 == 0:
            breach_ratio_high = None
        else:
            breach_ratio_high = (max_val - gp95) / gp95

        records.append({
            'cmdb_id': cmdb_id,
            'trace_name': trace_name,
            'anomaly_count_high': anomaly_count_high,
            'earliest_high_timestamp': earliest_high_ts,
            'anomaly_count_low': anomaly_count_low,
            'earliest_low_timestamp': earliest_low_ts,
            'max_value_in_window': max_val,
            'min_value_in_window': min_val,
            'global_p95': gp95,
            'global_p5': gp5,
            'breach_ratio_high': breach_ratio_high
        })

    trace_anom_df = pd.DataFrame.from_records(records)

    # 4) Sort by total anomalies desc, then breach_ratio_high desc, return top 20
    trace_anom_df['total_anoms'] = trace_anom_df['anomaly_count_high'] + trace_anom_df['anomaly_count_low']
    trace_anom_top20 = (
        trace_anom_df
        .sort_values(['total_anoms','breach_ratio_high'], ascending=[False, False], na_position='last')
        .drop(columns=['total_anoms'])
        .reset_index(drop=True)
    )

# Display compact result (top 20)
trace_anom_top20.head(20)

```
Out[3]:
```
# Summarize trace-level anomaly results as a plain-English string (uses existing `trace_anom_top20`)
import pandas as pd
import numpy as np

try:
    trace_anom_top20
except NameError:
    summary = "No trace anomaly results found in the environment."
else:
    total_groups = int(trace_anom_top20.shape[0])
    # compute total anomalies per row
    totals = (trace_anom_top20['anomaly_count_high'].fillna(0).astype(int)
              + trace_anom_top20['anomaly_count_low'].fillna(0).astype(int))
    groups_with_any = int((totals > 0).sum())
    # earliest high/low timestamps across reported groups
    def min_ts(col):
        s = trace_anom_top20[col].dropna()
        return s.min() if not s.empty else pd.NaT
    earliest_high = min_ts('earliest_high_timestamp')
    earliest_low = min_ts('earliest_low_timestamp')
    window_info = "2021-03-04 18:00:00+08:00 through 2021-03-04 18:30:00+08:00 (UTC+8)"
    
    # Build top lines (up to 10)
    lines = []
    top = trace_anom_top20.head(10).copy()
    def fmt(x, digits=6):
        if pd.isna(x):
            return "None"
        if isinstance(x, (pd.Timestamp, np.datetime64)):
            # isoformat preserves tz
            return pd.Timestamp(x).isoformat()
        if isinstance(x, (int, np.integer)):
            return str(int(x))
        return f"{float(x):.{digits}g}"
    for _, r in top.iterrows():
        lines.append(
            f"{r['cmdb_id']} | {r['trace_name']} | high={fmt(r['anomaly_count_high'])}, earliest_high={fmt(r['earliest_high_timestamp'])} | "
            f"low={fmt(r['anomaly_count_low'])}, earliest_low={fmt(r['earliest_low_timestamp'])} | "
            f"max={fmt(r['max_value_in_window'])}, min={fmt(r['min_value_in_window'])} | "
            f"p95={fmt(r['global_p95'])}, p5={fmt(r['global_p5'])} | breach_ratio_high={fmt(r['breach_ratio_high'])}"
        )
    
    summary = (
        f"Trace-level anomaly summary for window {window_info}:\n"
        f"- Candidate cmdb_ids checked: ['Redis01', 'IG01', 'Redis02', 'Mysql01', 'IG02']\n"
        f"- Total (cmdb_id, trace_name) groups returned: {total_groups}\n"
        f"- Groups with any anomalies (high or low): {groups_with_any}\n"
        f"- Earliest high-latency anomaly observed: {fmt(earliest_high)}\n"
        f"- Earliest low-throughput anomaly observed: {fmt(earliest_low)}\n\n"
        f"Top groups (up to 10) sorted by total anomalies:\n"
        + "\n".join(lines)
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  anomaly_count_high   earliest_high_timestamp  anomaly_count_low    earliest_low_timestamp  max_value_in_window  min_value_in_window  global_p95  global_p5  breach_ratio_high
0     IG02          trace.self.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:16:00+08:00             8.741750             0.134000    1.232577   0.339535           6.092252
1     IG01          trace.self.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:14:00+08:00             9.222000             0.442000    1.330012   0.562450           5.933769
2     IG02   trace.to_Tomcat02.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:09:00+08:00            11.064300             0.065250    1.614080   0.132510           5.854865
3     IG01  trace.to_Tomcat02.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:01:00+08:00             1.570125             0.067721    0.272372   0.073097           4.764633
4     IG01   trace.to_Tomcat02.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:06:00+08:00             9.515500             0.182250    1.663272   0.285940           4.720951
5     IG02  trace.to_Tomcat02.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:09:00+08:00             1.652737             0.051548    0.319224   0.063749           4.177361
6     IG01   trace.to_Tomcat04.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:14:00+08:00             5.759600             0.191200    1.119437   0.277873           4.145084
7     IG01   trace.to_Tomcat03.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:16:00+08:00             6.104500             0.117850    1.188725   0.185075           4.135334
8     IG01         trace.self.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:14:00+08:00             1.210556             0.088045    0.244466   0.090633           3.951839
9     IG01  trace.to_Tomcat04.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:14:00+08:00             1.160651             0.070161    0.246345   0.075233           3.711489
10    IG01   trace.to_Tomcat01.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:16:00+08:00             5.792000             0.097300    1.274760   0.391905           3.543600
11    IG02         trace.self.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:16:00+08:00             1.041765             0.060650    0.271337   0.087149           2.839376
12    IG02   trace.to_Tomcat04.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:12:00+08:00             4.898600             0.166100    1.276987   0.217800           2.836059
13    IG01  trace.to_Tomcat01.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:15:00+08:00             0.881000             0.049129    0.242899   0.079084           2.627020
14    IG02  trace.to_Tomcat04.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:16:00+08:00             0.964416             0.069250    0.293836   0.085645           2.282160
15    IG02   trace.to_Tomcat03.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:16:00+08:00             3.793100             0.065000    1.299757   0.121838           1.918314
16    IG01  trace.to_Tomcat03.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:16:00+08:00             0.760968             0.058845    0.299788   0.074346           1.538350
17    IG02   trace.to_Tomcat01.duration_p95                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:08:00+08:00             2.237300             0.086400    1.242730   0.110463           0.800311
18    IG02  trace.to_Tomcat03.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:16:00+08:00             0.496143             0.030969    0.302636   0.039843           0.639408
19    IG02  trace.to_Tomcat01.duration_mean                   2 2021-03-04 18:00:00+08:00                  2 2021-03-04 18:14:00+08:00             0.409520             0.049304    0.293049   0.062191           0.397445```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# ensure source DataFrames exist in the kernel
try:
    metric
    trace
    log
    error_logs
except NameError:
    # load if not present
    metric = pd.read_csv('metric.csv')
    trace = pd.read_csv('trace.csv')
    log = pd.read_csv('log.csv')
    error_logs = pd.read_csv('error_logs.csv')

# ensure numeric value and timezone-aware datetime column 'dt' (UTC+8)
for df in (metric, trace, log, error_logs):
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
    if 'dt' not in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# incident window (inclusive)
start = pd.Timestamp('2021-03-04 18:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-04 18:30:00').tz_localize(tz)

# 1) metric rows for IG01 with CPUWio or CPULoad or CPUSysTime
mask_metric = (
    (metric['cmdb_id'] == 'IG01') &
    (metric['dt'] >= start) & (metric['dt'] <= end) &
    metric['kpi_name'].str.contains('CPUWio|CPULoad|CPUSysTime', na=False)
)
metric_ig = (
    metric.loc[mask_metric, ['dt','cmdb_id','kpi_name','value']]
    .rename(columns={'dt':'timestamp'})
    .sort_values('timestamp', ascending=True)
    .head(50)
    .reset_index(drop=True)
)

# 2) trace rows for IG01 in window
mask_trace = (trace['cmdb_id'] == 'IG01') & (trace['dt'] >= start) & (trace['dt'] <= end)
trace_ig = (
    trace.loc[mask_trace, ['dt','cmdb_id','trace_name','value']]
    .rename(columns={'dt':'timestamp'})
    .sort_values('timestamp', ascending=True)
    .head(100)
    .reset_index(drop=True)
)

# 3) log rows for IG01 in window
mask_log = (log['cmdb_id'] == 'IG01') & (log['dt'] >= start) & (log['dt'] <= end)
log_ig = (
    log.loc[mask_log, ['dt','cmdb_id','log_name','value']]
    .rename(columns={'dt':'timestamp'})
    .sort_values('timestamp', ascending=True)
    .head(50)
    .reset_index(drop=True)
)

# 4) error_logs rows for IG01 in window
mask_err = (error_logs['cmdb_id'] == 'IG01') & (error_logs['dt'] >= start) & (error_logs['dt'] <= end)
error_ig = (
    error_logs.loc[mask_err, ['dt','cmdb_id','message']]
    .rename(columns={'dt':'timestamp'})
    .sort_values('timestamp', ascending=True)
    .head(50)
    .reset_index(drop=True)
)

# Display compact tables (variables returned)
metric_ig, trace_ig, log_ig, error_ig

```
Out[4]:
```
# Summarize IG01 telemetry within 2021-03-04 18:00–18:30 (UTC+8) as a plain-English string.
import pandas as pd

try:
    metric_ig, trace_ig, log_ig, error_ig
except NameError:
    summary = "Required dataframes (metric_ig, trace_ig, log_ig, error_ig) not found in the environment."
else:
    def ts_range(df):
        if df.shape[0] == 0:
            return "none"
        lo = df['timestamp'].min()
        hi = df['timestamp'].max()
        return f"{pd.Timestamp(lo).isoformat()} to {pd.Timestamp(hi).isoformat()}"
    
    metric_count = metric_ig.shape[0]
    metric_range = ts_range(metric_ig)
    # list sampled KPI names present (unique, up to 10)
    metric_kpis = ", ".join(metric_ig['kpi_name'].unique()[:10])
    
    trace_count = trace_ig.shape[0]
    trace_range = ts_range(trace_ig)
    trace_names = ", ".join(trace_ig['trace_name'].unique()[:10])
    
    log_count = log_ig.shape[0]
    err_count = error_ig.shape[0]
    
    summary = (
        "IG01 telemetry within 2021-03-04 18:00–18:30 (UTC+8):\n"
        f"- Metrics: {metric_count} rows returned (filtered to kpi_name containing CPUWio/CPULoad/CPUSysTime), "
        f"time range = {metric_range}. Sample KPI names: {metric_kpis}.\n"
        f"- Traces: {trace_count} rows returned (all IG01 traces), time range = {trace_range}. "
        f"Sample trace names: {trace_names}.\n"
        f"- Logs: {log_count} rows found for IG01 in the window (none returned).\n"
        f"- Error logs: {err_count} rows found for IG01 in the window (none returned).\n\n"
        "Notes: Tables were limited (metrics ≤50 rows, traces ≤100 rows, logs/errors ≤50 rows)."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp cmdb_id                                   kpi_name     value
0  2021-03-04 18:00:00+08:00    IG01  JVM-Operating System_7778_JVM_JVM_CPULoad  0.244700
1  2021-03-04 18:00:00+08:00    IG01                    OSLinux-CPU_CPU_CPULoad  1.000000
2  2021-03-04 18:00:00+08:00    IG01                 OSLinux-CPU_CPU_CPUSysTime  0.299300
3  2021-03-04 18:00:00+08:00    IG01                     OSLinux-CPU_CPU_CPUWio  2.386400
4  2021-03-04 18:01:00+08:00    IG01                    OSLinux-CPU_CPU_CPULoad  1.000000
..                       ...     ...                                        ...       ...
45 2021-03-04 18:12:00+08:00    IG01                 OSLinux-CPU_CPU_CPUSysTime  0.257775
46 2021-03-04 18:12:00+08:00    IG01                     OSLinux-CPU_CPU_CPUWio  0.157991
47 2021-03-04 18:13:00+08:00    IG01                     OSLinux-CPU_CPU_CPUWio  0.020800
48 2021-03-04 18:13:00+08:00    IG01  JVM-Operating System_7778_JVM_JVM_CPULoad  0.206500
49 2021-03-04 18:13:00+08:00    IG01                    OSLinux-CPU_CPU_CPULoad  1.020000

[50 rows x 4 columns],                    timestamp cmdb_id                       trace_name       value
0  2021-03-04 18:00:00+08:00    IG01         trace.self.duration_mean    1.210556
1  2021-03-04 18:00:00+08:00    IG01      trace.to_Tomcat04.row_count   63.000000
2  2021-03-04 18:00:00+08:00    IG01   trace.to_Tomcat04.duration_p95    5.759600
3  2021-03-04 18:00:00+08:00    IG01  trace.to_Tomcat04.duration_mean    1.160651
4  2021-03-04 18:00:00+08:00    IG01      trace.to_Tomcat03.row_count   62.000000
..                       ...     ...                              ...         ...
95 2021-03-04 18:06:00+08:00    IG01  trace.to_Tomcat03.duration_mean    0.095731
96 2021-03-04 18:06:00+08:00    IG01      trace.to_Tomcat02.row_count   98.000000
97 2021-03-04 18:06:00+08:00    IG01   trace.to_Tomcat02.duration_p95    0.250300
98 2021-03-04 18:06:00+08:00    IG01      trace.to_Tomcat01.row_count  100.000000
99 2021-03-04 18:06:00+08:00    IG01   trace.to_Tomcat01.duration_p95    1.029050

[100 rows x 4 columns], Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing 'metric' if present, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure numeric value and dt (timezone-aware UTC+8)
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
if 'dt' not in metric.columns:
    metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure dt has correct tz (in case)
    metric['dt'] = pd.to_datetime(metric['dt']).dt.tz_localize(None).dt.tz_localize('UTC').dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id='IG01', kpi_name) using full series (do BEFORE filtering)
global_p95_ig = (
    metric[metric['cmdb_id'] == 'IG01']
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric to incident window and IG01 rows
start = pd.Timestamp('2021-03-04 18:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-04 18:30:00').tz_localize(tz)

metric_ig_window = metric[
    (metric['cmdb_id'] == 'IG01') &
    (metric['dt'] >= start) & (metric['dt'] <= end)
].copy()

# Merge global p95 into windowed rows
metric_ig_window = metric_ig_window.merge(global_p95_ig, on=['cmdb_id','kpi_name'], how='left')

# 3) For each kpi_name find consecutive anomalous segments where value > global_p95
segments = []
for kpi, grp in metric_ig_window.groupby('kpi_name'):
    g = grp.sort_values('dt').reset_index(drop=True)
    # If global_p95 missing for this kpi (shouldn't happen), skip
    if g['global_p95'].isna().all():
        continue
    # Flag anomalies
    g['is_anom'] = g['value'] > g['global_p95']
    # Keep only anomaly rows to compute consecutive segments
    anom = g[g['is_anom']].copy()
    if anom.shape[0] < 2:
        continue  # need at least 2 for a segment
    # Compute time diff in seconds between consecutive anomaly rows
    anom['dt_diff_s'] = anom['dt'].diff().dt.total_seconds().fillna(np.inf)
    # A new segment starts when dt_diff_s != 60
    anom['segment_id'] = (anom['dt_diff_s'] != 60).cumsum()
    # Aggregate per segment
    seg_agg = (
        anom
        .groupby('segment_id', as_index=False)
        .agg(
            cmdb_id = ('cmdb_id', 'first'),
            kpi_name = ('kpi_name', 'first'),
            segment_start_timestamp = ('dt', 'min'),
            segment_end_timestamp = ('dt', 'max'),
            length_in_minutes = ('dt', lambda s: int(((s.max() - s.min()).total_seconds() / 60) + 1)),  # inclusive count
            max_value_in_segment = ('value', 'max'),
            global_p95 = ('global_p95', 'first')
        )
    )
    # Filter segments with length >= 2 minutes (i.e., at least two consecutive anomaly timestamps)
    seg_agg = seg_agg[seg_agg['length_in_minutes'] >= 2].copy()
    if seg_agg.empty:
        continue
    # Compute breach_ratio, handle zero or NaN global_p95
    def br(row):
        gp = row['global_p95']
        if pd.isna(gp) or gp == 0:
            return None
        return (row['max_value_in_segment'] - gp) / gp
    seg_agg['breach_ratio'] = seg_agg.apply(br, axis=1)
    segments.append(seg_agg)

# Combine all segments
if segments:
    segments_df = pd.concat(segments, ignore_index=True)
    # 5) Sort by length desc then breach_ratio desc, keep top 20 and select requested columns
    segments_df = segments_df.sort_values(['length_in_minutes','breach_ratio'], ascending=[False, False], na_position='last')
    result_segments = segments_df.loc[:, [
        'kpi_name','segment_start_timestamp','segment_end_timestamp',
        'length_in_minutes','max_value_in_segment','global_p95','breach_ratio'
    ]].reset_index(drop=True).head(20)
else:
    # empty result DataFrame with specified columns
    result_segments = pd.DataFrame(columns=[
        'kpi_name','segment_start_timestamp','segment_end_timestamp',
        'length_in_minutes','max_value_in_segment','global_p95','breach_ratio'
    ])

# Display compact result
result_segments

```
Out[5]:
```
# Summarize the IG01 consecutive-anomaly segment detection result as plain English.
try:
    result_segments
except NameError:
    summary = "IG01 consecutive-anomaly check: No result available in the environment."
else:
    if result_segments.shape[0] == 0:
        summary = (
            "Conclusion: No consecutive anomalous segments (>= 2 consecutive minutes where value > global P95) "
            "were found for IG01 metrics within 2021-03-04 18:00–18:30 (UTC+8). "
            "Method: global P95 per IG01 kpi computed from the full series, window filtered, and consecutive-minute "
            "(60s gap) anomaly segments extracted. None met the length>=2 criterion. "
            "If you want single-minute anomalies or all anomaly timestamps per KPI, I can provide those."
        )
    else:
        # brief summary of top segments if any (not expected here)
        top = result_segments.head(5)
        lines = []
        for _, r in top.iterrows():
            lines.append(
                f"{r['kpi_name']}: {r['segment_start_timestamp'].isoformat()} to {r['segment_end_timestamp'].isoformat()}, "
                f"len={int(r['length_in_minutes'])}m, max={float(r['max_value_in_segment']):.6g}, p95={float(r['global_p95']):.6g}"
            )
        summary = (
            "Detected consecutive anomalous segments for IG01 (top up to 5 shown):\n" +
            "\n".join(lines)
        )

summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [kpi_name, segment_start_timestamp, segment_end_timestamp, length_in_minutes, max_value_in_segment, global_p95, breach_ratio]
Index: []```
```